# Data Cleaning — NorthStar

Applies the cleaning actions from Section 2.3 of the report. Nulls and anomalies are kept and paired with flag columns. Zones are standardised across all files using one shared mapping.

## 1. Setup

In [ ]:
import pandas as pd
from pathlib import Path

raw_dir     = Path('../data/raw')
cleaned_dir = Path('../data/cleaned')
cleaned_dir.mkdir(parents=True, exist_ok=True)

ZONE_MAPPING = {
    'North': 'North', 'NORTH': 'North', 'north': 'North',
    'South': 'South', 'SOUTH': 'South',
    'East':  'East',  'EAST':  'East',
    'West':  'West',  'WEST':  'West',
    'Central': 'Central', 'CENTRAL': 'Central', 'Ctr': 'Central',
    'Airport': 'Airport', 'AIRPORT': 'Airport',
    'Riverside': 'Riverside', 'RIVERSIDE': 'Riverside', 'RiverSide': 'Riverside',
}
CANONICAL_ZONES = sorted(set(ZONE_MAPPING.values()))
print(f'Canonical zones: {CANONICAL_ZONES}')

## 2. customers

In [ ]:
customers = pd.read_csv(raw_dir / 'customers.csv')
customers['home_zone'] = customers['home_zone'].map(ZONE_MAPPING).fillna(customers['home_zone'])
customers['loyalty_score_missing'] = customers['loyalty_score'].isna().astype(int)
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
customers.to_csv(cleaned_dir / 'customers_cleaned.csv', index=False)
print('home_zone unique:', sorted(customers['home_zone'].unique()))

## 3. orders

In [ ]:
orders = pd.read_csv(raw_dir / 'orders.csv')
for col in ['pickup_zone', 'dropoff_zone']:
    orders[col] = orders[col].map(ZONE_MAPPING).fillna(orders[col])

priority = pd.CategoricalDtype(categories=['Low', 'Medium', 'High', 'Critical'], ordered=True)
orders['priority_level']   = orders['priority_level'].astype(priority)
orders['order_created_at'] = pd.to_datetime(orders['order_created_at'], errors='coerce')
orders.to_csv(cleaned_dir / 'orders_cleaned.csv', index=False)
print('pickup_zone unique:', sorted(orders['pickup_zone'].unique()))

## 4. deliveries

In [ ]:
deliveries = pd.read_csv(raw_dir / 'deliveries.csv')
deliveries['dispatch_time']         = pd.to_datetime(deliveries['dispatch_time'], errors='coerce')
deliveries['delivery_completed_at'] = pd.to_datetime(deliveries['delivery_completed_at'], errors='coerce')

deliveries['temporal_anomaly_flag'] = (
    deliveries['delivery_completed_at'].notna()
    & (deliveries['delivery_completed_at'] < deliveries['dispatch_time'])
).astype(int)

deliveries['status_timestamp_conflict'] = (
    (deliveries['delivery_status'] == 'OnTime')
    & deliveries['delivery_completed_at'].isna()
).astype(int)

deliveries.to_csv(cleaned_dir / 'deliveries_cleaned.csv', index=False)
print('temporal anomalies:', deliveries['temporal_anomaly_flag'].sum())
print('status/timestamp conflicts:', deliveries['status_timestamp_conflict'].sum())

## 5. drivers

In [ ]:
drivers = pd.read_csv(raw_dir / 'drivers.csv')
drivers['base_zone'] = drivers['base_zone'].map(ZONE_MAPPING).fillna(drivers['base_zone'])
drivers['training_score_missing'] = drivers['training_score'].isna().astype(int)
drivers.to_csv(cleaned_dir / 'drivers_cleaned.csv', index=False)
print('missing training scores:', drivers['training_score_missing'].sum())

## 6. vehicles

In [ ]:
vehicles = pd.read_csv(raw_dir / 'vehicles.csv')
vehicles['assigned_zone']   = vehicles['assigned_zone'].map(ZONE_MAPPING).fillna(vehicles['assigned_zone'])
vehicles['commission_date'] = pd.to_datetime(vehicles['commission_date'], errors='coerce')
vehicles.to_csv(cleaned_dir / 'vehicles_cleaned.csv', index=False)

## 7. complaints

In [ ]:
complaints = pd.read_csv(raw_dir / 'complaints.csv')
sev = pd.CategoricalDtype(categories=['Low', 'Medium', 'High'], ordered=True)
complaints['severity']   = complaints['severity'].astype(sev)
complaints['created_at'] = pd.to_datetime(complaints['created_at'], errors='coerce')
complaints.to_csv(cleaned_dir / 'complaints_cleaned.csv', index=False)

## 8. incidents

In [ ]:
incidents = pd.read_csv(raw_dir / 'incidents.csv')
sev = pd.CategoricalDtype(categories=['Low', 'Medium', 'High', 'Critical'], ordered=True)
incidents['severity']    = incidents['severity'].astype(sev)
incidents['reported_at'] = pd.to_datetime(incidents['reported_at'], errors='coerce')
incidents.to_csv(cleaned_dir / 'incidents_cleaned.csv', index=False)

## 9. app_events

In [ ]:
app_events = pd.read_csv(raw_dir / 'app_events.csv')
app_events['zone_context']    = app_events['zone_context'].map(ZONE_MAPPING).fillna(app_events['zone_context'])
app_events['event_timestamp'] = pd.to_datetime(app_events['event_timestamp'], errors='coerce')
app_events.to_csv(cleaned_dir / 'app_events_cleaned.csv', index=False)

## 10. hubs

In [ ]:
hubs = pd.read_csv(raw_dir / 'hubs.csv')
hubs.to_csv(cleaned_dir / 'hubs_cleaned.csv', index=False)
print('hub rows:', len(hubs))

## 11. Integrity check

Make sure every primary key is unique and every foreign key resolves before exporting.

In [ ]:
pk_checks = [
    ('customers', 'customer_id', customers),
    ('orders', 'order_id', orders),
    ('deliveries', 'delivery_id', deliveries),
    ('drivers', 'driver_id', drivers),
    ('vehicles', 'vehicle_id', vehicles),
    ('complaints', 'complaint_id', complaints),
    ('incidents', 'incident_id', incidents),
    ('app_events', 'event_id', app_events),
    ('hubs', 'hub_id', hubs),
]
for name, col, df in pk_checks:
    dup = df[col].duplicated().sum()
    print(f'{name} {col}: {dup} duplicates')
    assert dup == 0

In [ ]:
cust_ids  = set(customers['customer_id'])
order_ids = set(orders['order_id'])
drv_ids   = set(drivers['driver_id'])
veh_ids   = set(vehicles['vehicle_id'])
hub_ids   = set(hubs['hub_id'])

fk_checks = [
    ('orders.customer_id',      (~orders['customer_id'].isin(cust_ids)).sum()),
    ('deliveries.order_id',     (~deliveries['order_id'].isin(order_ids)).sum()),
    ('deliveries.driver_id',    (~deliveries['driver_id'].isin(drv_ids)).sum()),
    ('deliveries.vehicle_id',   (~deliveries['vehicle_id'].isin(veh_ids)).sum()),
    ('deliveries.hub_id',       (~deliveries['hub_id'].isin(hub_ids)).sum()),
    ('complaints.customer_id',  (~complaints['customer_id'].isin(cust_ids)).sum()),
    ('complaints.order_id',     (~complaints['order_id'].isin(order_ids)).sum()),
]
for name, broken in fk_checks:
    print(f'{name}: {broken} broken FK')